# 06: TextWorld memory at action selection

The live runner retrieves retained descriptions, inventory and action records before choosing each command. Its scripted selector follows the generated game's visible ordered instructions, matching the next remembered instruction to an admissible command. It reconstructs progress from retrieved action records and falls back to an untried command when no instruction matches. The optional model receives the current observation and the same retrieved records in its prompt.

When TextWorld is unavailable, the notebook explicitly replays its existing fixed trace. That path records storage and inventory survival; its supplied success and reward are not evidence that memory changed an action.

Outputs include operation logs, per-turn retrieved IDs and payloads, commands, cumulative score and incremental reward credit. The runner has been exercised with TextWorld 1.7.0. Read each saved run with its original source revision: the [notebook guide](README.md#select-a-live-prune-with-an-episode-replay) distinguishes the earlier exploration selector, repaired controller and completed live-retention diagnostic. This notebook's four-policy comparison and separate read-boundary probe do not themselves run that diagnostic.


## 1. Setup


In [ ]:
RUN_MODE = "demo"  # @param ["demo", "paper"]
POLICY_BACKEND = "scripted"  # @param ["scripted", "ollama"]


In [ ]:
import sys
import json
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = NOTEBOOK_DIR.parent
sys.path.insert(0, str(NOTEBOOK_DIR))

from _utils import (
    NotebookConfig,
    print_config,
    print_repro_stamp,
    set_all_seeds,
    RunManager,
    bootstrap_ci,
    paired_permutation_test,
    effect_size,
    setup_plotting,
    add_caption,
    COLORS,
    POLICY_COLORS,
)
from _utils.adapters import (
    textworld_adapter,
    ollama_healthcheck,
    ollama_chat,
    OLLAMA_AVAILABLE,
    parse_llm_action,
)

from reckon.core.resources import MemoryBudget
from reckon.memory import CausalMemoryManager, PayloadType
from reckon.memory import FIFOPolicy, LRUPolicy, UtilityPolicy, EvidencePerBytePolicy

import matplotlib.pyplot as plt
import numpy as np

setup_plotting()
print("Imports OK!")


In [ ]:
config = NotebookConfig(
    notebook_name="06_textworld_adapter",
    run_mode=RUN_MODE,
    seed=42,
    seed_count=15 if RUN_MODE == "paper" else 5,
    budget_slots=25,
    task_name="textworld",
    num_episodes=15 if RUN_MODE == "paper" else 5,
    evidence_weight=0.5,
    custom={
        "policy_backend": POLICY_BACKEND,
        "fixture_version": 3,
    },
)
print_config(config)
print_repro_stamp(config)
set_all_seeds(config.seed)


In [ ]:
TEXTWORLD_AVAILABLE = textworld_adapter.check_available()
print(
    f"TextWorld available: {TEXTWORLD_AVAILABLE}; requested backend: {POLICY_BACKEND}"
)
if not TEXTWORLD_AVAILABLE:
    print(
        "Replaying a fixed trace. Its supplied reward is not a memory-dependent game outcome."
    )
    print(textworld_adapter.get_install_command())
print(
    f"Ollama available: {OLLAMA_AVAILABLE}; failed or unavailable model calls are counted."
)


## 2. Questions

Text games combine room descriptions, available commands and inventory. The task asks whether retained records can affect the next command, and what is lost when those records are pruned. The scripted selector needs the visible quest description and the retained action history to reconstruct progress. When instruction matching fails, remembered commands help it avoid immediate repetition. The optional model also sees retrieved descriptions and inventory.

Entity tags are regex-derived labels. Tag counts and inventory-record survival are observations about stored data, not extraction accuracy or proof that a game was solved. The current comparison includes assigned priorities and learned utility; it does not attach measured intervention effects.


## 3. The memory loop

At reset the runner records the description and available inventory. At each decision it retrieves descriptions, inventory and action records within a fresh episode context. The scripted selector reads ordered instructions from the current observation or a retained description and advances through them using retrieved actions; without a matching next instruction it tries a command absent from that history; the model prompt contains the current observation, admissible commands and the exact retrieved payloads.

TextWorld returns a cumulative score. The runner credits only its change since the previous step, then records the action, next description, inventory and reward change. Success reads the requested `won` flag, not merely a positive score. Model parsing failures or unavailable calls use the same memory-fed scripted fallback and increment `fallback_count`. That count measures model-to-script fallback, not the scripted selector's own fallback to an untried command.

The adapter wraps the native environment with [TextWorld's Filter](https://textworld.readthedocs.io/en/stable/_modules/textworld/envs/wrappers/filter.html), which supplies the observation/info tuple used here. It seeds generated games through [GameOptions](https://textworld.readthedocs.io/en/stable/textworld.gym.html) and the underlying environment. A reproducible live comparison still needs the generated game, dependency versions and any model sampling parameters retained with the run.


## 4. System Mapping

```mermaid
flowchart TB
    subgraph ENV["TextWorld Environment"]
        E1["textworld.start(game)"]
        E2["obs, info = env.reset()"]
        E3["obs, reward, done, info = env.step(command)"]
    end

    subgraph ADAPTER["TextworldAdapter"]
        A1["record_description()"]
        A2["record_action()"]
        A3["record_inventory()"]
        A4["record_reward()"]
        A5["_extract_entities() - regex"]
    end

    subgraph POLICY["Action Selection"]
        P1["Scripted: follow remembered instructions, then try an untried command"]
        P2["Ollama: LLM chooses from admissible"]
    end

    subgraph CMM["CausalMemoryManager"]
        M1["write() with extracted tags"]
        M2["Retrieve descriptions, inventory and actions"]
        M3["prune() under budget"]
    end

    ENV --> ADAPTER
    ADAPTER --> CMM
    CMM --> POLICY
    ENV --> POLICY
    POLICY --> ENV
```


## 5. Experiment Design

**Setup:**

- Game: Treasure hunt (3 rooms, 5 objects) or demo trace
- Episodes: 5 (demo) / 15 (paper)
- Controller: Visible-instruction heuristic with an untried-command fallback, or Ollama

**Comparison:**

- Policies: FIFO, LRU, Utility, Evidence per byte
- Budgets: 5, 10, 15, 25 slots

**Metrics:**

- Retained prerecorded entity-tag counts (inspection, not extraction accuracy)
- Inventory retention rate
- Requested won flag (live game); supplied success (fixed trace)
- Records per episode

The tag-count section replays the fixed demo's supplied entity tags and counts those still present in memory. It has no labelled correctness comparison or denominator for extraction accuracy. Evaluating the live regex extractor requires labelled observations and an explicit accuracy measure.

This comparison limits record slots only. Whole text observations can exceed a byte allowance scaled from a few hundred bytes per slot; records are not silently truncated. Each result reports `byte_budget=None` and actual final stored bytes. Native byte-budget comparisons remain available through the command-line experiment runner.


## 6. Demo Run


In [ ]:
from _utils.adapters.text_game import run_textworld_episode

# Run demo
demo_result = run_textworld_episode(
    config.seed,
    policy_backend=POLICY_BACKEND,
    verbose=True,
)
print(f"Episode result:")
print(f"  Success: {demo_result['success']}")
print(f"  Steps: {demo_result['steps']}")
print(f"  Records stored: {demo_result['records']}")
print("  Inventory retention:", demo_result["inventory_retention"])
print(f"  Demo mode: {demo_result['demo_mode']}")


In [ ]:
# Analyze demo trace
print("Demo trace analysis:")
trace = textworld_adapter.get_demo_trace()

for event in trace:
    step = event["step"]
    etype = event["type"]

    if etype == "description":
        entities = event.get("entities", [])
        print(f"  Step {step}: {etype} - entities={entities}")
    elif etype == "action":
        cmd = event.get("command", "")
        print(f"  Step {step}: {etype} - '{cmd}'")
    elif etype == "inventory":
        items = event.get("items", [])
        print(f"  Step {step}: {etype} - {items}")
    elif etype == "reward":
        reward = event.get("reward", 0)
        print(f"  Step {step}: {etype} - {reward}")


## 7. Full Run


In [ ]:
# Policy and budget configurations
POLICIES = {
    "FIFO": FIFOPolicy(),
    "LRU": LRUPolicy(),
    "Utility": UtilityPolicy(),
    "Evidence per byte": EvidencePerBytePolicy(config.evidence_weight),
}

BUDGETS = [5, 10, 15, 25]

print(f"Running {config.num_episodes} episodes per configuration")
print(f"Policies: {list(POLICIES.keys())}")
print(f"Budgets: {BUDGETS}")
print(f"Policy backend: {POLICY_BACKEND}")


In [ ]:
# Prepare one generated case per seed, then give every policy that same file.
import time

case_started = time.perf_counter()
case_files = {}
if TEXTWORLD_AVAILABLE:
    for seed in range(config.seed, config.seed + config.num_episodes):
        adapter = textworld_adapter.TextworldAdapter(
            textworld_adapter.TextworldConfig(seed=seed)
        )
        try:
            adapter.make_env()
            case_files[seed] = adapter.game_file
        finally:
            adapter.close()
case_preparation_seconds = time.perf_counter() - case_started

# Run experiments
all_results = []
all_transcripts = []

total_runs = len(POLICIES) * len(BUDGETS) * config.num_episodes
run_count = 0

for policy_name, policy in POLICIES.items():
    for budget in BUDGETS:
        for ep in range(config.num_episodes):
            seed = config.seed + ep
            result = run_textworld_episode(
                seed=seed,
                budget_slots=budget,
                policy=policy,
                policy_backend=POLICY_BACKEND,
                game_file=case_files.get(seed),
            )

            # Store result (without events)
            result_summary = {
                k: v for k, v in result.items() if k not in ["events", "transcript"]
            }
            all_results.append(result_summary)
            all_transcripts.append(
                {
                    "policy": policy_name,
                    "budget": budget,
                    "seed": seed,
                    "environment": result["environment"],
                    "transcript": result["transcript"],
                }
            )

            run_count += 1
            if run_count % 30 == 0:
                print(f"Progress: {run_count}/{total_runs}")

print(f"\nCompleted {len(all_results)} runs")


In [ ]:
from statistics import mean, stdev


def scores_for(policy_name, budget, metric):
    return [
        r[metric]
        for r in all_results
        if r["policy"] == policy_name
        and r["budget"] == budget
        and r[metric] is not None
    ]


summary = []
for policy in POLICIES.values():
    for budget in BUDGETS:
        row = {"policy": type(policy).__name__, "budget": budget}
        for metric in ["success", "inventory_retention", "steps", "records"]:
            values = scores_for(type(policy).__name__, budget, metric)
            row[metric] = mean(values) if values else None
        summary.append(row)
        print(row)


## 8. Results


In [ ]:
# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Success rate
ax = axes[0]
for policy_name in POLICIES.keys():
    values = [
        scores_for(type(POLICIES[policy_name]).__name__, budget, "success")
        for budget in BUDGETS
    ]
    means = [mean(v) if v else float("nan") for v in values]
    stds = [stdev(v) if len(v) > 1 else 0 for v in values]

    color = POLICY_COLORS.get(policy_name.lower(), COLORS["neutral"])
    ax.errorbar(
        BUDGETS,
        means,
        yerr=stds,
        fmt="o-",
        label=policy_name,
        color=color,
        linewidth=2,
        markersize=8,
        capsize=4,
    )

ax.set_xlabel("Memory Budget (slots)")
ax.set_ylabel("Success Rate")
ax.set_ylim(0, 1.05)
ax.legend()
ax.set_title("Success Rate vs Budget")

# Right: Inventory retention
ax = axes[1]
for policy_name in POLICIES.keys():
    values = [
        scores_for(type(POLICIES[policy_name]).__name__, budget, "inventory_retention")
        for budget in BUDGETS
    ]
    means = [mean(v) if v else float("nan") for v in values]
    stds = [stdev(v) if len(v) > 1 else 0 for v in values]

    color = POLICY_COLORS.get(policy_name.lower(), COLORS["neutral"])
    ax.errorbar(
        BUDGETS,
        means,
        yerr=stds,
        fmt="o-",
        label=policy_name,
        color=color,
        linewidth=2,
        markersize=8,
        capsize=4,
    )

ax.set_xlabel("Memory Budget (slots)")
ax.set_ylabel("Inventory Record Retention Rate")
ax.set_ylim(0, 1.05)
ax.legend()
ax.set_title("Inventory Record Retention vs Budget")

add_caption(
    axes[0],
    f"Inventory survival; inspect environment and action path",
    n=config.num_episodes * len(BUDGETS),
    mode=RUN_MODE,
)

plt.tight_layout()
plt.show()


In [ ]:
# Inspect retained prerecorded entity tags; this does not evaluate the live extractor.
budget = MemoryBudget(max_slots=25, max_bytes=25 * 300)
cmm = CausalMemoryManager(budget, EvidencePerBytePolicy(0.5), "analysis")
textworld_adapter.run_demo_episode(cmm, config.seed)

records = cmm.get_all_records()

# Count entity tags
entity_tags = ["key", "door", "chest", "treasure", "room", "corridor"]
entity_counts = {tag: 0 for tag in entity_tags}

for r in records:
    for tag in r.tags:
        if tag in entity_counts:
            entity_counts[tag] += 1

print("Retained prerecorded entity-tag counts (not accuracy):")
for tag, count in sorted(entity_counts.items(), key=lambda x: -x[1]):
    if count > 0:
        print(f"  {tag}: {count} occurrences")

# The live control changes only whether retained context reaches each decision.
context_controls = []
if TEXTWORLD_AVAILABLE:
    for use_memory in [True, False]:
        result = run_textworld_episode(
            config.seed,
            100,
            policy=FIFOPolicy(),
            policy_backend="scripted",
            game_file=case_files.get(config.seed),
            use_memory=use_memory,
        )
        assert not any(event["op_type"] == "prune" for event in result["events"])
        assert any(turn["retrieved_ids"] for turn in result["transcript"]) == use_memory
        context_controls.append(result)
    assert context_controls[0]["game_identity"] == context_controls[1]["game_identity"]
    print("Matched live-game controls:", [r["game_identity"] for r in context_controls])
    print(
        "Commands with context:",
        [t["command"] for t in context_controls[0]["transcript"]],
    )
    print(
        "Commands without context:",
        [t["command"] for t in context_controls[1]["transcript"]],
    )


In [ ]:
print(
    "Fixed fallback traces are not independent trials. Live pairing requires retained generated-game identity and model settings before inferential comparisons are enabled."
)


## 9. Debugger Walkthrough

After exporting the demo's `logs/events.jsonl`, follow the [native debugger quick start](../src/reckon/viz/README.md#quick-start). The notebook inspects entity tags and retained inventory records directly; operation logs expose write/read/prune IDs/details rather than the full descriptions, commands or entity tags.

Keep that distinction when inspecting the demo run. Its log is not every comparison episode, not a proof that a local-model action used memory, and not pair-bound graph evidence.


## 10. Limits and remaining checks

The scripted selector recognizes a narrow generated instruction grammar. Lost descriptions or action history can erase its task or progress, and compound instructions can be misread even with full memory. Its untried-command fallback can also block a legitimate repeated command in another room. The [completed diagnostic](runs/2026-11-16/textworld_retention_diagnostic/6676e6a9ecd8/summary.md) preserves these observed limits, including a parsing failure and a fallback win; it establishes no general text-game capability. The optional model sees retained context but can be slow or stochastic. Its failures are counted, not silently relabelled as model actions.

The fixed fallback trace ignores the seed and always replays its supplied rewards. Its inventory fraction measures actual remaining records. It does not exercise live command selection. Protocol fixtures exercise the real memory manager with a test adapter; they do not establish that the native TextWorld engine or a live model runs.

The live adapter uses the documented Filter wrapper, seeds generation and preserves cumulative-score semantics. The saved live runs retain generated-game identities and separate scripted-action outcomes. A live model comparison still needs its model configuration and execution evidence. Regex entity extraction still needs labelled observations if it is to be evaluated for accuracy.

Generated games are retained once under `runs/textworld_games/`; each run lists its binary names and SHA-256 digests for both executable and JSON metadata. Their JSON sidecars keep full game logic and text, with the machine-local default grammar directory cleared. TextWorld loads its installed default grammar when regeneration needs it. Custom supplied game files are not rewritten.


## Remove one record at its first read

This separate experiment runs fresh seed-matched episodes under FIFO with enough storage to reach the selected read. It removes one logical record immediately before its first eligible retrieval, verifies matching record state in both arms, and charges both episodes and the full evaluation time. The target is the first retained action record returned by a read in this TextWorld game. Later replacement records remain allowed. These measurements do not score the main comparison's live store or implement measured retention.

The interval is a normal approximation across paired episode differences; the sign-permutation test is unadjusted across record labels. Four demo pairs and twenty paper-mode pairs demonstrate execution; neither count was selected for statistical power. The probe requires the actual game and scripted controller; model-backed and prerecorded fallback runs do not supply this paired evidence. An absent or ambiguous target raises an error rather than becoming a zero effect. TextWorld's scripted controller can still produce zero game wins in both arms.

In [ ]:
from _utils.adapters.record_probe import AdapterProbeConfig, evaluate_adapter_record

record_probe = None
if TEXTWORLD_AVAILABLE and POLICY_BACKEND == "scripted":
    record_probe = evaluate_adapter_record(
        AdapterProbeConfig("textworld", budget_slots=100, max_steps=30),
        "first_action",
        lambda record: "action" in record.tags,
        n_trials=20 if RUN_MODE == "paper" else 4,
        seed_base=1000,
    )
    print({key: record_probe[key] for key in ["protocol", "probe_episodes", "probe_elapsed_s", "confidence_interval"]})
    print({key: value for key, value in record_probe["evaluation"].items() if key != "trials"})
else:
    print("Record probe unavailable: paired evidence requires the declared simulated or live scripted controller.")

## Export Artifacts


In [ ]:
run = RunManager(config)
run.save_config()
if record_probe is not None:
    (run.run_dir / "record_probe.json").write_text(json.dumps(record_probe, indent=2) + "\n")
run.save_metrics(all_results)
run.save_jsonl(all_transcripts, "transcripts.jsonl")
run.save_jsonl(context_controls, "context_controls.jsonl")

# Save event logs
if demo_result.get("events"):
    run.save_logs(demo_result["events"])

# Save figure
fig, ax = plt.subplots(figsize=(8, 5))
for policy_name in POLICIES.keys():
    values = [
        scores_for(type(POLICIES[policy_name]).__name__, budget, "inventory_retention")
        for budget in BUDGETS
    ]
    means = [mean(v) if v else float("nan") for v in values]
    color = POLICY_COLORS.get(policy_name.lower(), COLORS["neutral"])
    ax.plot(BUDGETS, means, "o-", label=policy_name, color=color)
ax.set_xlabel("Budget")
ax.set_ylabel("Inventory Record Retention")
ax.legend()
run.save_figure(fig, "textworld_retention")
plt.close(fig)

# Save environment metadata
env_meta = {
    "textworld_available": TEXTWORLD_AVAILABLE,
    "ollama_available": OLLAMA_AVAILABLE,
    "policy_backend": POLICY_BACKEND,
    "policies": list(POLICIES.keys()),
    "budgets": BUDGETS,
}
with open(run.run_dir / "env_meta.json", "w") as f:
    json.dump(env_meta, f, indent=2)

import hashlib
import importlib.metadata
import subprocess

source_paths = [
    NOTEBOOK_DIR / "_utils/adapters/record_probe.py",
    NOTEBOOK_DIR / "_utils/adapters/episode_memory.py",
    NOTEBOOK_DIR / "06_text_env_adapter_textworld.ipynb",
    NOTEBOOK_DIR / "_utils/adapters/text_game.py",
    NOTEBOOK_DIR / "_utils/adapters/textworld_adapter.py",
    NOTEBOOK_DIR / "_utils/artifacts.py",
]
source = {
    "run_mode": RUN_MODE,
    "textworld_version": (
        importlib.metadata.version("textworld") if TEXTWORLD_AVAILABLE else None
    ),
    "uncommitted_source": bool(
        subprocess.run(
            [
                "git",
                "status",
                "--porcelain",
                "--",
                *[str(path) for path in source_paths],
            ],
            capture_output=True,
            text=True,
            check=True,
        ).stdout
    ),
    "sha256": {
        str(path.relative_to(NOTEBOOK_DIR)): hashlib.sha256(
            path.read_bytes()
        ).hexdigest()
        for path in source_paths
    },
}
(run.run_dir / "source.json").write_text(json.dumps(source, indent=2) + "\n")
# Games are shared experiment inputs under notebooks/runs/textworld_games.
games = {
    r["game_identity"]["file"]: {
        key: value for key, value in r["game_identity"].items() if key != "file"
    }
    for r in all_results + ([arm["result"] for arm in record_probe["arms"]] if record_probe else [])
    if r.get("game_identity")
}
(run.run_dir / "games.json").write_text(json.dumps(games, indent=2) + "\n")
# Summary
inventory_rates = [
    r["inventory_retention"]
    for r in all_results
    if r["inventory_retention"] is not None
]
run.write_summary(
    {
        "run_mode": RUN_MODE,
        "policy_backend": POLICY_BACKEND,
        "comparison_episodes": len(all_results),
        "case_preparation_seconds": case_preparation_seconds,
        "context_control_episodes": len(context_controls),
        "record_probe_episodes": record_probe["probe_episodes"] if record_probe else 0,
        "record_probe_elapsed_s": record_probe["probe_elapsed_s"] if record_probe else 0,
        "record_probe_delta": record_probe["evaluation"]["delta"] if record_probe else None,
        "record_probe_available": record_probe is not None,
        "inspection_episodes": 1,
        "fixed_trace_inspections": 1,
        "textworld_available": TEXTWORLD_AVAILABLE,
        "mean_success_rate": mean(r["success"] for r in all_results),
        "mean_inventory_retention": (
            mean(inventory_rates) if inventory_rates else None
        ),
    }
)

run.print_artifacts()
